# We will make a bried EDA and create data quality and dictionary

### We will import and read the data first

In [21]:
import pandas as pd

df = pd.read_parquet("../data/final/board_golden.parquet")

df.head()

,ticker,exchange,person_name,role,source_agreement,confidence_score,merged_at
0,ACB,HOSE,Ông Bùi Tấn Tài,Phó TGĐ,conflict,0.8,2026-03-05T01:24:06.173808
1,ACB,HOSE,Bà Dương Thị Nguyệt,KTT,conflict,0.8,2026-03-05T01:24:06.173808
2,ACB,HOSE,Ông Hiep Van Vo,TVHĐQT,vietstock_only,0.6,2026-03-05T01:24:06.173808
3,ACB,HOSE,Bà Hoàng Ngân,Thành viên BKS,both,1.0,2026-03-05T01:24:06.173808
4,ACB,HOSE,Ông Huỳnh Nghĩa Hiệp,Trưởng BKS,both,1.0,2026-03-05T01:24:06.173808


### Check the data info and create dictionary

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 942 entries, 0 to 941
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ticker            942 non-null    object 
 1   exchange          942 non-null    object 
 2   person_name       942 non-null    object 
 3   role              942 non-null    object 
 4   source_agreement  942 non-null    object 
 5   confidence_score  942 non-null    float64
 6   merged_at         942 non-null    object 
dtypes: float64(1), object(6)
memory usage: 51.6+ KB


In [23]:
from datetime import datetime

descriptions = {
    "ticker": "Stock ticker symbol of the listed company.",
    "exchange": "Stock exchange where the company is listed (HOSE, HNX, UPCOM).",
    "person_name": "Full name of the board member as displayed on source.",
    "role": "Board/leadership role in Vietnamese.",
    "source_agreement": "Indicates whether the record appears in both sources, only one source, or contains conflicting information.",
    "confidence_score": "Confidence level assigned based on source agreement (1.0=both, 0.8=conflict, 0.6=single source).",
    "merged_at": "Timestamp indicating when the merge pipeline was executed."
}

lines = []

for col in df.columns:
    dtype = df[col].dtype
    desc = descriptions.get(col, "N/A")
    lines.append(f"| {col} | {dtype} | {desc} |")

dictionary = f"""
# Data Dictionary

| Column | Data Type | Description |
|--------|-----------|-------------|
{chr(10).join(lines)}
"""

with open("../docs/data_dictionary.md", "w") as f:
    f.write(dictionary)

print("Data dictionary saved.")

Data dictionary saved.


### Check the quality and create data quality summary

#### check null 

In [24]:
null_summary = pd.DataFrame({
    "null_count": df.isna().sum(),
    "null_rate": df.isna().mean()
}).sort_values("null_rate", ascending=False)

null_summary

,null_count,null_rate
ticker,0,0.0
exchange,0,0.0
person_name,0,0.0
role,0,0.0
source_agreement,0,0.0
confidence_score,0,0.0
merged_at,0,0.0


#### check uniqueness

In [25]:
print("Unique tickers:", df["ticker"].nunique())
print("Unique persons:", df["person_name"].nunique())

print("\nSource agreement distribution:")
print(df["source_agreement"].value_counts())

Unique tickers: 59
Unique persons: 893

Source agreement distribution:
source_agreement
conflict          413
vietstock_only    286
cafef_only        154
both               89
Name: count, dtype: int64


#### Check match and conflict rate

In [26]:
print("\nSource agreement distribution:")
print(df["source_agreement"].value_counts())


Source agreement distribution:
source_agreement
conflict          413
vietstock_only    286
cafef_only        154
both               89
Name: count, dtype: int64


In [27]:
total_records = len(df)

agreement_counts = df["source_agreement"].value_counts()

both_count = agreement_counts.get("both", 0)
conflict_count = agreement_counts.get("conflict", 0)

match_rate = both_count / total_records
conflict_rate = conflict_count / total_records

print("\nMatching metrics:")
print("Match rate (both sources): {:.2%}".format(match_rate))
print("Conflict rate: {:.2%}".format(conflict_rate))


Matching metrics:
Match rate (both sources): 9.45%
Conflict rate: 43.84%


#### Unmatch name

In [28]:
# Top unmatched names

cafef_only = df[df["source_agreement"] == "cafef_only"]
vietstock_only = df[df["source_agreement"] == "vietstock_only"]

print("\nTop 10 CafeF-only names:")
print(cafef_only["person_name"].value_counts().head(10))

print("\nTop 10 Vietstock-only names:")
print(vietstock_only["person_name"].value_counts().head(10))


Top 10 CafeF-only names:
person_name
Ông Nguyễn Anh Tuấn      3
Ông Nguyễn Thanh Toại    1
Bà Phạm Thị Thanh Nga    1
Bà Phan Lạc Kim Trinh    1
Ông Từ Quốc Học          1
Ông Đặng Xuân Thắng      1
Ông Quách Kiều Hưng      1
Bà Lê Thị Hồng Nhung     1
Bà Trần Thị Nhung Gấm    1
Ông Phạm Hồng Hà         1
Name: count, dtype: int64

Top 10 Vietstock-only names:
person_name
Ông Nguyễn Anh Tuấn      3
Ông Nguyễn Duy Khánh     2
Ông Nguyễn Việt Thắng    2
Ông Nguyễn Ngọc Quang    2
*** ***                  2
Bà Bùi Thu Hà            1
Ông Hoàng Xuân Quốc      1
Ông Lê Chiến Thắng       1
Ông Lê Cự Tân            1
Ông Nguyễn Tuấn Hùng     1
Name: count, dtype: int64


#### Create data quality summary

In [29]:
null_lines = []

for col, row in null_summary.iterrows():
    null_count = row["null_count"]
    null_rate = row["null_rate"]
    null_lines.append(
        f"| {col} | {null_count} | {null_rate:.2%} |"
    )

agreement_lines = []

agreement_counts = df["source_agreement"].value_counts()

for label, count in agreement_counts.items():
    agreement_lines.append(f"| {label} | {count} |")

null_section = "\n".join(null_lines)
agreement_section = "\n".join(agreement_lines)

cafef_counts = cafef_only["person_name"].value_counts().head(10)
vietstock_counts = vietstock_only["person_name"].value_counts().head(10)

cafef_top_table = "\n".join(
    [f"| {name} | {count} |" for name, count in cafef_counts.items()]
)

vietstock_top_table = "\n".join(
    [f"| {name} | {count} |" for name, count in vietstock_counts.items()]
)

In [30]:
report = f"""
# Data Quality Report

## Total Records
{len(df)}

## Total Columns
{len(df.columns)}

## Unique Counts
- Unique tickers: {df["ticker"].nunique()}
- Unique persons: {df["person_name"].nunique()}

## Null Summary

| Column | Null Count | Null Rate |
|--------|------------|-----------|
{null_section}

## Source Agreement Distribution

| Category | Count |
|----------|-------|
{agreement_section}

## Matching Metrics

- Match rate (both sources): {match_rate:.2%}
- Conflict rate: {conflict_rate:.2%}

## Unmatched Names

### CafeF Only (Top 10)

| Name | Count |
|------|-------|
{cafef_top_table}

### Vietstock Only (Top 10)

| Name | Count |
|------|-------|
{vietstock_top_table}

## Observations

- The match rate indicates partial overlap between CafeF and Vietstock.
- The conflict rate suggests differences in role naming conventions or update timing.
- Vietstock contains more single-source records than CafeF.
- Name normalization significantly improves matching quality.
- Further improvements could include fuzzy matching for higher alignment.
"""

with open("../docs/data_quality_report.md", "w") as f:
    f.write(report)

print("Data quality report saved.")

Data quality report saved.
